# Results for model: baichuan-inc/baichuan2-13b-chat

In [ ]:
# Import required libraries
import polars as pl
import xgboost as xgb

# Load train data
train_data = pl.read_parquet('./jane_street_data/train.parquet')

# Feature engineering: Calculate a global TOP_QUANTILE (top 15%) of 'feature_00'
# relative to 'responder_6' across rolling batches of 'date_id'.
def top_quantile(data, feature_name, target_column, window_size=100):
    # Calculate rolling batch means of feature_00 and calc_top_quantile
    rolled_mean = data.groupby('date_id')[feature_name].mean().rolling(window_size).mean()
    calc_top_quantile = rolled_mean.orderBy(rolled_mean.desc(), asc_by='date_id').head(int(rolled_mean.collect().size * 0.15))
    
    # Join partitions, making sure that the feature and target columns are always paired
    paired_data = data.join(
        rolled_mean.join(
            calc_top_quantile.relax(constant_window_bounds=True),
            on='date_id',
            how='inner').to_pandas(),
        on='date_id',
        how='inner')
    
    # Create the new calculated feature
    paired_data[f"{feature_name}.top_quantile"] = paired_data[feature_name] * calculated_top_quantile
    
    # Return the new updated data
    return paired_data

# Apply feature engineering
train_data = top_quantile(train_data, 'feature_00', 'responder_6')

# Train XGBoost Regressor on the target 'responder_6'
X_train = train_data.to_pandas().drop('responder_6', axis=1)
y_train = train_data['responder_6']

# Use 70% of the data for training, 30% for validation
train_data = xgb.DMatrix(X_train, label=y_train)

# Train the model
xgb_params = {
    'max_depth': 4,
    'eta': 0.1,
    'silent': 1,
    'objective': 'reg:linear',
    'eval_metric': 'rmse',
    'early_stopping_rounds': 100,
    'num_boost_round': 500
}

model = xgb.train(xgb_params, train_data, valid_data=train_data)

# Save the model
model.save_model('jane_street_model.dat')